<a href="https://colab.research.google.com/github/adnanjopo-collab/Olist-Ecommerce-Analysis/blob/main/Olist_Ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================
# OLIST BRAZILIAN E-COMMERCE - DATA ANALYTICS
# Author: Adnan Mustafa
# Tools: Python, MySQL (Aiven Console), Power BI
# ================================================

from google.colab import drive
drive.mount('/content/drive')

import zipfile

# Extract dataset from zip file
zip_path = '/content/drive/MyDrive/Brazilian E Commerce Public Dataset by Olist/archive.zip'
extract_path = '/content/olist/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files extracted successfully!


In [ ]:
import pandas as pd

# Load all CSV files
orders = pd.read_csv('/content/olist/olist_orders_dataset.csv')
customers = pd.read_csv('/content/olist/olist_customers_dataset.csv')
order_items = pd.read_csv('/content/olist/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/content/olist/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/olist/olist_order_reviews_dataset.csv')
products = pd.read_csv('/content/olist/olist_products_dataset.csv', encoding='latin-1')
sellers = pd.read_csv('/content/olist/olist_sellers_dataset.csv', encoding='latin-1')
category_translation = pd.read_csv('/content/olist/product_category_name_translation.csv')

# Verify shapes
datasets = {
    'orders': orders, 'customers': customers,
    'order_items': order_items, 'order_payments': order_payments,
    'order_reviews': order_reviews, 'products': products,
    'sellers': sellers, 'category_translation': category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [ ]:
# Check null values in each dataset
print("=== NULL VALUES ===")
for name, df in datasets.items():
    null_count = df.isnull().sum().sum()
    print(f"{name}: {null_count} null values")

# Check which columns have null values
print("\n=== NULL COLUMNS DETAIL ===")
for name, df in [('orders', orders),
                  ('order_reviews', order_reviews),
                  ('products', products)]:
    print(f"\n--- {name} ---")
    null_cols = df.isnull().sum()
    print(null_cols[null_cols > 0])

# Check order status distribution
print("\n=== ORDER STATUS ===")
print(orders['order_status'].value_counts())

# Check duplicates
print("\n=== DUPLICATES ===")
for name, df in [('orders', orders), ('customers', customers),
                  ('order_items', order_items), ('order_payments', order_payments),
                  ('products', products), ('sellers', sellers)]:
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

# Check data types
print("\n=== ORDERS DATA TYPES ===")
print(orders.dtypes)

=== NULL VALUES ===
orders: 4908 null values
customers: 0 null values
order_items: 0 null values
order_payments: 0 null values
order_reviews: 145903 null values
products: 2448 null values
sellers: 0 null values
category_translation: 0 null values

=== NULL COLUMNS DETAIL ===

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- order_reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

=== ORDER STATUS ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created          

In [ ]:
# ---- Orders: Convert date columns to datetime ----
date_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# ---- Orders: Map customer_id to customer_unique_id ----
customer_id_map = customers.set_index('customer_id')['customer_unique_id']
orders['customer_unique_id'] = orders['customer_id'].map(customer_id_map)

# ---- Products: Fill missing category with Unknown ----
products['product_category_name'] = products['product_category_name'].fillna('Unknown')

# ---- Products: Drop 2 rows where weight is null ----
products_clean = products.dropna(subset=['product_weight_g'])

# ---- Products: Translate category names to English ----
products_clean = products_clean.merge(
    category_translation, on='product_category_name', how='left'
)
products_clean['product_category_name'] = (
    products_clean['product_category_name_english'].fillna('Unknown')
)
products_clean = products_clean.drop(columns=['product_category_name_english'])

# ---- Verify cleaning results ----
print("=== CLEANING RESULTS ===")
print(f"Orders shape: {orders.shape}")
print(f"Products: {products.shape} -> {products_clean.shape}")
print(f"Null customer_unique_id: {orders['customer_unique_id'].isnull().sum()}")
print(f"\nSample customer mapping:")
print(orders[['customer_id', 'customer_unique_id']].head())
print(f"\nSample delivery dates:")
print(orders['order_delivered_customer_date'].value_counts().tail())
print(orders.columns.tolist())

=== CLEANING RESULTS ===
Orders shape: (99441, 9)
Products: (32951, 9) -> (32949, 9)
Null customer_unique_id: 0

Sample customer mapping:
                        customer_id                customer_unique_id
0  9ef432eb6251297304e76186b10a928d  7c396fd4830fd04220f754e42b4e5bff
1  b0830fb4747a6c6d20dea0b8c802d7ef  af07308b275d755c9edb36a90c618231
2  41ce2a54c0b03bf3443c3d931a367089  3a653a41f6f9fc3d2a113cf8398680e8
3  f88197465ea7920adcdbec7375364d82  7c142cf63193a1473d2e66489a9ae977
4  8ab97904e6daea8866dbdbc4fb7aad2c  72632f0f9dd73dfee390c9b22eb56dd6

Sample delivery dates:
order_delivered_customer_date
2018-01-17 13:29:13    1
2018-06-29 11:21:34    1
2018-01-09 21:42:59    1
2018-06-19 17:02:46    1
2018-06-07 16:38:35    1
Name: count, dtype: int64
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id']


In [ ]:
!pip install pymysql -q
from sqlalchemy import create_engine, text

# --- PROFESSIONAL MAPPING: BRAZIL STATE CODES TO FULL NAMES ---
brazil_states_map = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AM': 'Amazonas', 'AP': 'Amapá',
    'BA': 'Bahia', 'CE': 'Ceará', 'DF': 'Distrito Federal', 'ES': 'Espírito Santo',
    'GO': 'Goiás', 'MA': 'Maranhão', 'MG': 'Minas Gerais', 'MS': 'Mato Grosso do Sul',
    'MT': 'Mato Grosso', 'PA': 'Pará', 'PB': 'Paraíba', 'PE': 'Pernambuco',
    'PI': 'Piauí', 'PR': 'Paraná', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RO': 'Rondônia', 'RR': 'Roraima', 'RS': 'Rio Grande do Sul', 'SC': 'Santa Catarina',
    'SE': 'Sergipe', 'SP': 'São Paulo', 'TO': 'Tocantins'
}

# --- STEP 1: DATA PREPARATION ---
payment_info = order_payments.sort_values(by=['order_id', 'payment_value'], ascending=[True, False])
payment_info = payment_info.drop_duplicates(subset=['order_id'])

# Dimension: Customer_Lookup
Customer_Lookup = customers[['customer_unique_id', 'customer_city', 'customer_state']].rename(columns={
    'customer_unique_id': 'CustomerID', 'customer_city': 'City', 'customer_state': 'StateCode'
}).drop_duplicates(subset=['CustomerID'])
Customer_Lookup['State'] = Customer_Lookup['StateCode'].map(brazil_states_map) + ", Brazil"
Customer_Lookup = Customer_Lookup.drop(columns=['StateCode'])

# Dimension: Seller_Lookup
Seller_Lookup = sellers[['seller_id', 'seller_city', 'seller_state']].rename(columns={
    'seller_id': 'SellerID', 'seller_city': 'City', 'seller_state': 'StateCode'
})
Seller_Lookup['State'] = Seller_Lookup['StateCode'].map(brazil_states_map) + ", Brazil"
Seller_Lookup = Seller_Lookup.drop(columns=['StateCode'])

# Dimension: Product_Lookup
Product_Lookup = products_clean[['product_id', 'product_category_name']].rename(columns={
    'product_id': 'ProductID', 'product_category_name': 'Category'
})

# Dimension: Reviews_Lookup
Reviews_Lookup = order_reviews[['order_id', 'review_score', 'review_creation_date']].rename(columns={
    'order_id': 'OrderID', 'review_score': 'ReviewScore', 'review_creation_date': 'ReviewDate'
})
Reviews_Lookup['ReviewDate'] = pd.to_datetime(Reviews_Lookup['ReviewDate'])
Reviews_Lookup = Reviews_Lookup.sort_values('ReviewScore', ascending=False).drop_duplicates(subset=['OrderID'])

# Fact: Sales_Data
Sales_Data = order_items.merge(
    orders[['order_id', 'customer_unique_id', 'order_status', 'order_purchase_timestamp',
            'order_delivered_customer_date', 'order_estimated_delivery_date']],
    on='order_id', how='left'
).merge(
    payment_info[['order_id', 'payment_type', 'payment_value']], on='order_id', how='left'
).rename(columns={
    'order_id': 'OrderID', 'customer_unique_id': 'CustomerID', 'order_status': 'OrderStatus',
    'order_purchase_timestamp': 'OrderDate', 'order_delivered_customer_date': 'DeliveryDate',
    'order_estimated_delivery_date': 'EstimatedDeliveryDate', 'payment_type': 'PaymentType',
    'payment_value': 'PaymentValue', 'product_id': 'ProductID', 'seller_id': 'SellerID',
    'price': 'Price', 'freight_value': 'FreightValue', 'shipping_limit_date': 'ShippingLimitDate'
})
Sales_Data['PaymentType'] = Sales_Data['PaymentType'].fillna('Unknown')

# Integrity Check & Primary Key Generation
Sales_Data = Sales_Data[Sales_Data['ProductID'].isin(Product_Lookup['ProductID'])]
Reviews_Lookup = Reviews_Lookup[Reviews_Lookup['OrderID'].isin(Sales_Data['OrderID'])]

for df in [Sales_Data, Customer_Lookup, Product_Lookup, Seller_Lookup, Reviews_Lookup]:
    df.insert(0, 'RowID', range(1, len(df) + 1))

# --- STEP 2: SECURE DATABASE UPLOAD (WITH PRIMARY KEY ENFORCEMENT) ---
engine = create_engine("mysql+pymysql://Your Credentials")

def upload_with_pk(table_name, df, engine):
    """Creates table with an explicit Primary Key and uploads data."""
    print(f"Uploading {table_name}... ({len(df):,} rows)")

    # Generate SQL Column Definitions
    cols = []
    for col, dtype in df.dtypes.items():
        if col == 'RowID':
            cols.append(f"`{col}` INT NOT NULL PRIMARY KEY")
        elif 'int' in str(dtype).lower():
            cols.append(f"`{col}` BIGINT")
        elif 'float' in str(dtype).lower():
            cols.append(f"`{col}` DOUBLE")
        elif 'datetime' in str(dtype).lower():
            cols.append(f"`{col}` DATETIME")
        else:
            cols.append(f"`{col}` TEXT") # Use TEXT for strings to avoid length issues

    create_sql = f"CREATE TABLE IF NOT EXISTS `{table_name}` ({', '.join(cols)}) ENGINE=InnoDB;"

    with engine.connect() as conn:
        conn.execute(text(f"DROP TABLE IF EXISTS `{table_name}`;"))
        conn.execute(text(create_sql))
        conn.commit()

    # Append data to the now-existing table with PK
    df.to_sql(name=table_name, con=engine, if_exists='append', index=False, chunksize=10000)
    print(f"Success: {table_name} uploaded with Primary Key.")

final_tables = {
    'Sales_Data': Sales_Data, 'Customer_Lookup': Customer_Lookup,
    'Product_Lookup': Product_Lookup, 'Seller_Lookup': Seller_Lookup,
    'Reviews_Lookup': Reviews_Lookup
}

for name, df in final_tables.items():
    upload_with_pk(name, df, engine)

print("\nETL PROCESS COMPLETED SUCCESSFULLY!")

Uploading Sales_Data... (112,632 rows)
Success: Sales_Data uploaded with Primary Key.
Uploading Customer_Lookup... (96,096 rows)
Success: Customer_Lookup uploaded with Primary Key.
Uploading Product_Lookup... (32,949 rows)
Success: Product_Lookup uploaded with Primary Key.
Uploading Seller_Lookup... (3,095 rows)
Success: Seller_Lookup uploaded with Primary Key.
Uploading Reviews_Lookup... (97,901 rows)
Success: Reviews_Lookup uploaded with Primary Key.

ETL PROCESS COMPLETED SUCCESSFULLY!
